In [4]:
import pandas as pd
import numpy as np
import json

In [29]:
data_dir = "../../../data/llm_performance/annotation_stuff"
expr_df = pd.read_csv(f"{data_dir}/all_nonprag_expressions.csv")
expr_df['old_id'] = expr_df['gen_passage_id'].apply(lambda x: x.replace('olmo1', 'ai').replace('olmo2', 'ai'))
expr_df.head(1)

,gen_passage_id,annotator,expression,note,old_id
0,18037_olmo1,celesteamidon,suggested a mind hardened by the weight of aut...,NaN,18037_ai


In [30]:
paras_for_few_shot = ['15031_ai', '16704_ai', '16204_ai']

In [31]:
all_paras_df = pd.read_csv(f"{data_dir}/all_passages.csv")
print(all_paras_df.shape)
all_paras_df.head(2)

(100, 2)


,gen_passage_id,passage
0,18137_human,"A couple of months before, Fletcher told me, h..."
1,18090_human,Now Caro and Grace Bell did not go home at onc...


In [32]:
# convert olmo1, olmo2 to ai in old ids
all_paras_df['old_id'] = all_paras_df['gen_passage_id'].apply(lambda x: x.replace('olmo1', 'ai').replace('olmo2', 'ai'))

In [33]:
# sample 57 for training, 40 for testing, keep the 3 few shot as training 
train_size = 60
test_val_size = 100-train_size
train_paras = (all_paras_df[~all_paras_df['old_id']
                            .isin(paras_for_few_shot)]
                            .sample(train_size-3, random_state=42)
                            ['old_id'].tolist() 
                            + paras_for_few_shot)
assert len(set(train_paras)) == train_size

test_val_paras = (all_paras_df[~all_paras_df['old_id']
                           .isin(paras_for_few_shot) 
                           & ~all_paras_df['old_id']
                           .isin(train_paras)]['old_id'].tolist())
assert len(set(test_val_paras)) == test_val_size

for p in paras_for_few_shot:
    assert p in train_paras
    assert p not in test_val_paras

# split into test and val
# np.random.seed(42)
# np.random.shuffle(test_val_paras)
# test_paras = test_val_paras[:test_val_size//2]
# val_paras = test_val_paras[test_val_size//2:]
# assert len(set(test_paras).intersection(set(val_paras))) == 0
# print(len(train_paras), len(val_paras), len(test_paras))

In [34]:
# save a idx: gen_passage_id dict
out_data_dir = "../../../data/llm_performance/ft/train_test"
id2para = {i: pid for i, pid in enumerate(test_val_paras)}
with open(f"{out_data_dir}/prag_test_id2para.json", "w") as f:
    json.dump(id2para, f, indent=4)

id2para = {i: pid for i, pid in enumerate(train_paras)}
with open(f"{out_data_dir}/prag_train_id2para.json", "w") as f:
    json.dump(id2para, f, indent=4)

In [38]:
for data_split, paras in zip(["train", "test"], 
                                  [train_paras, test_val_paras]):
    num_expressions = 0
    examples  = []
    for passage_id in paras:
        para_text = all_paras_df[all_paras_df['old_id'] == passage_id]['passage'].values[0]
        para_nov_df = expr_df[expr_df['old_id'] == passage_id].copy()
        para_nov_df = para_nov_df.drop_duplicates(subset=['expression'])
        para_expressions = para_nov_df['expression'].tolist()
        
        # print(len(para_expressions))
        para_justifications = para_nov_df['note'].tolist()
        input = f"Passage:\n{para_text}\n"
        input += "Non-pragmatic expressions in json format:"
        num_expressions += len(para_expressions)
        if len(para_expressions) == 0:
            output = "\n[]\n\n"
        else:
            output = "\n["
            fs_jsons = []
            for expression, justification in zip(para_expressions, para_justifications):
                if  pd.isna(justification) or justification.strip() == "":
                    justification = "The expression does not make sense contextually."
                fs_jsons.append({"expression": expression.strip(), "justification": justification.strip()})
            output += ",\n".join([json.dumps(j, ensure_ascii=False) for j in fs_jsons]) + "]"

        example = {"messages": [{"role": "user", "content": input},
                            {"role": "assistant", "content": output}],}
        examples.append(example)
    print(f"{data_split}: {len(paras)} passages, {num_expressions} expressions")
    with open(f"{out_data_dir}/prag_{data_split}.jsonl", "w") as f:
        for example in examples:
            f.write(json.dumps(example, ensure_ascii=False) + "\n")


train: 60 passages, 291 expressions
test: 40 passages, 244 expressions


In [47]:
print(examples[3]['messages'][1]['content'])


[{"expression": "she inherited from her mother had cooled off all Alisa’s passionate impulses", "justification": "This clause is a little awkward and not clear."},
{"expression": "she was always passionately in love and suffered spectacularly", "justification": "In love with what or who? This is not clear."},
{"expression": "She was always ready to bring to her new lover’s feet everything she owned", "justification": "Why would she bring things to her lover's feet? This doesn't seem to make sense contextually."},
{"expression": "thereby inflicting the deadly blow", "justification": "Martha's already dead, so her last lover can't inflict a deadly blow."},
{"expression": "By the time life was brought to perfection", "justification": "The use of passive voice here (\"was brought\") is confusing."},
{"expression": "The last costly touch was a small bathtub", "justification": "What makes the bathtub costly?"},
{"expression": "to the point of the psych wards", "justification": "A bit of a s